# SQL Queries with Pandas

**Goal:** use Pandas to inspect, filter, combine, and summarize data loaded from the existing PostgreSQL `imdb` database.

This notebook focuses on **Pandas data-management operations**. SQL appears in comments as a reference. The notebook only issues `SELECT` statements: it does not create, update, or delete database objects or records.

**Before starting:** PostgreSQL must be running, the `imdb` database and `imdb` schema must exist, and your account must have permission to read the tables used in the lesson.

## Learning path and suggested pacing

1. **A short Pandas introduction** — understand `DataFrame` and `Series`, and inspect a dataset.
2. Load the database tables, then select columns and create derived values.
3. Filter, handle missing values, search text, and sort results.
4. Combine tables with joins and identify missing relationships.
5. Reproduce set operations and aggregate queries.
6. Translate selected subqueries into Pandas and solve the final exercises.

Four short exercises are placed after the relevant sections; the final section contains broader practice tasks.

## 1. Introduction to Pandas

Pandas is a Python library for working with labelled, tabular data. Its two main objects are:

- A **`DataFrame`** is a table with rows and named columns. Columns can have different data types.
- A **`Series`** is one column of data, with an index that labels its rows.

A DataFrame lives in Python's memory; it is not a database table. In this lesson, we first load query results from PostgreSQL and then perform transformations locally with Pandas.

In [ ]:
import pandas as pd

# Create a small in-memory example; no database is needed for this cell.
sample_movies = pd.DataFrame(
    {
        "title": ["Arrival", "Moonlight", "The Martian"],
        "year": [2016, 2016, 2015],
        "length": [116, 111, 144]
    }
)

sample_movies

Selecting one column by name returns a `Series`; selecting a list of columns returns a `DataFrame`. This distinction is useful because many Pandas methods can be applied either to a whole table or to a single column.

In [ ]:
# A Series: one column from the DataFrame.
sample_movies["title"]

Before analysing a dataset, inspect its **shape**, column names, and data types. These checks help you understand which data is available and spot issues such as a year stored as text. `describe()` gives a compact statistical summary of numeric columns.

In [ ]:
# Basic inspection; these are DataFrame metadata/summary operations, rather than a single SQL query.
print("Shape (rows, columns):", sample_movies.shape)
print("Columns:", list(sample_movies.columns))
print("Data types:")
print(sample_movies.dtypes)
sample_movies.describe()

Official reference for this short introduction: [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html).

## 2. Connect to PostgreSQL and load the tables

Pandas performs its transformations in Python **after** data has been loaded. SQLAlchemy manages the database connection, and `pd.read_sql_query()` runs a SQL query and returns its result as a DataFrame.

The connection settings below assume a local PostgreSQL server. Change the user, host, port, or database name to match your setup.

In [ ]:
import sqlalchemy
from sqlalchemy import URL


DB_USER = "postgres"
DB_PASSWORD = ""  # Enter your password if required.
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "imdb"
connection_url = URL.create(
    "postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = sqlalchemy.create_engine(connection_url)

In [ ]:
# Load only the tables used by the examples and exercises.
with engine.connect() as connection:
    movie = pd.read_sql_query("SELECT * FROM imdb.movie", connection)
    person = pd.read_sql_query("SELECT * FROM imdb.person", connection)
    crew = pd.read_sql_query("SELECT * FROM imdb.crew", connection)
    genre = pd.read_sql_query("SELECT * FROM imdb.genre", connection)
    produced = pd.read_sql_query("SELECT * FROM imdb.produced", connection)
    country = pd.read_sql_query("SELECT * FROM imdb.country", connection)
    rating = pd.read_sql_query("SELECT * FROM imdb.rating", connection)
    location = pd.read_sql_query("SELECT * FROM imdb.location", connection)
    released = pd.read_sql_query("SELECT * FROM imdb.released", connection)

print(f"Movies: {len(movie):,}; people: {len(person):,}; crew records: {len(crew):,}")

## 3. Select columns, create derived columns, and remove duplicates

Use a list of column names inside double brackets to select several columns. `assign()` creates a derived column from an expression applied to a whole Series. This is a **vectorized operation**: Pandas applies the operation to all rows without a Python loop. `drop_duplicates()` removes repeated values or rows, depending on whether it is applied to a Series or DataFrame.

In [ ]:
# SQL:
# SELECT id, official_title, year FROM imdb.movie;
movie[["id", "official_title", "year"]]

In [ ]:
# SQL:
# SELECT official_title, length / 60.0 AS length_hours FROM imdb.movie;
movie.assign(length_hours=movie["length"] / 60.0)[["official_title", "length_hours"]]

In [ ]:
# SQL:
# SELECT DISTINCT genre FROM imdb.genre;
genre["genre"].drop_duplicates().reset_index(drop=True)

## 4. Filter rows with conditions

A **Boolean mask** is a Series of `True` and `False` values. Use it to keep only rows where a condition is true. `.loc[rows, columns]` is a clear way to specify both the row condition and the output columns.

Common equivalents:

- `==`, `!=`, `>`, `>=`, `<`, `<=`: comparisons.
- `&` / `|`: AND / OR. Put parentheses around **each** condition.
- `.between(low, high)`: values between the endpoints, inclusive by default, like SQL `BETWEEN`.
- `.isin([...])`: membership in a list, like SQL `IN (...)`.
- `~mask`: negates a Boolean mask.

In Pandas, use `&` and `|` for element-by-element Boolean combinations; Python's `and` and `or` are not substitutes for these operations on Series.

In [ ]:
# SQL:
# SELECT id, official_title, year, length
# FROM imdb.movie
# WHERE year = '2010' AND length > 60;
movie.loc[
    (movie["year"] == "2010") & (movie["length"] > 60),
    ["id", "official_title", "year", "length"]
]

In [ ]:
# SQL:
# SELECT id, official_title, year, length
# FROM imdb.movie
# WHERE year IN ('2010', '2011', '2012') AND length BETWEEN 60 AND 120;
movie.loc[
    movie["year"].isin(["2010", "2011", "2012"]) & movie["length"].between(60, 120),
    ["id", "official_title", "year", "length"]
]

## 5. Missing values

In SQL, ordinary comparisons with `NULL` do not evaluate to `TRUE`; use `IS NULL` or `IS NOT NULL`. In Pandas, missing data may appear as `None`, `NaN`, or `pd.NA`, depending on the column's data type. Use `.isna()` and `.notna()` to test for missing values; do not use `== None`.

**Important:** Pandas comparisons involving missing values can behave differently from SQL's three-valued logic. When filtering, rows whose condition is missing are not selected. `Series.count()` ignores missing values, and `groupby()` drops missing group keys by default unless `dropna=False` is supplied. We will revisit the grouping rule in the aggregation section.

In [ ]:
# SQL:
# SELECT id, given_name, death_date
# FROM imdb.person
# WHERE death_date IS NULL;
person.loc[person["death_date"].isna(), ["id", "given_name", "death_date"]]

In [ ]:
# SQL:
# SELECT id, given_name, death_date
# FROM imdb.person
# WHERE death_date IS NOT NULL;
person.loc[person["death_date"].notna(), ["id", "given_name", "death_date"]]

## 6. Text matching and normalization

Pandas string methods are vectorized: `.str.contains()` searches across an entire Series. `case=False` makes a search case-insensitive, `na=False` treats missing text as a non-match, and `regex=False` treats the search string literally.

For exact comparisons, `.str.lower()` or `.str.upper()` can normalize letter case, while `.str.strip()` removes leading and trailing whitespace. Unlike SQL `LIKE`, which uses `%` and `_` as wildcards, `.str.contains()` uses regular expressions by default; use `regex=False` for a literal phrase.

In [ ]:
# SQL (PostgreSQL):
# SELECT id, official_title
# FROM imdb.movie
# WHERE official_title ILIKE '%family%';
movie.loc[
    movie["official_title"].str.contains("family", case=False, na=False, regex=False),
    ["id", "official_title"]
]

In [ ]:
# SQL:
# SELECT id, official_title
# FROM imdb.movie
# WHERE LOWER(TRIM(official_title)) = 'inception';
movie.loc[
    movie["official_title"].str.strip().str.lower() == "inception",
    ["id", "official_title"]
]

## 7. Sort rows and limit the result

Use `.sort_values()` to sort by one or more columns. The `ascending` argument can be a single Boolean or a list of Booleans corresponding to the sort columns. `.head(n)` returns the first `n` rows after the sort, similar to PostgreSQL `LIMIT n`.

**Note:** Pandas puts missing sort values last by default, even in descending order. PostgreSQL defaults to `NULLS LAST` for ascending order and `NULLS FIRST` for descending order, so the results can differ when sort keys are missing.

In [ ]:
# SQL:
# SELECT *
# FROM imdb.person
# ORDER BY birth_date ASC, given_name DESC
# LIMIT 5;
person.sort_values(["birth_date", "given_name"], ascending=[True, False]).head()

### Exercise 1 — Filtering and sorting

Select the title, year, and length of movies released in 2010 that are at least 90 minutes long. Sort from longest to shortest.

In [ ]:
(
    movie.loc[
        (movie["year"] == "2010") & (movie["length"] >= 90),
        ["official_title", "year", "length"]
    ]
    .sort_values("length", ascending=False)
)

## 8. Combine tables with `merge()` (SQL joins)

SQL joins connect records through related keys. Pandas uses `merge()` for the same purpose:

- `how="inner"` keeps only matching rows (`INNER JOIN`). This is the default.
- `how="left"` keeps every row in the left DataFrame and adds matching data from the right (`LEFT JOIN`).
- `how="right"` and `how="outer"` correspond to `RIGHT JOIN` and `FULL OUTER JOIN`. A right join can be rewritten as a left join by swapping the inputs.
- `left_on` and `right_on` specify the matching columns when the key names differ.

**Important:** a join can produce several rows for one movie if it has several related records. Do not remove duplicates automatically: use `drop_duplicates()` only when the intended SQL result uses `DISTINCT` or when unique rows are required.

In [ ]:
# Link movies with their production country

In [ ]:
# SQL:
# SELECT movie.id, movie.official_title, produced.country
# FROM imdb.movie
# INNER JOIN imdb.produced ON movie.id = produced.movie;
movie_productions = movie.merge(produced, left_on="id", right_on="movie")
movie_productions[["id", "official_title", "country"]]

### Join three tables

To list movies featuring a named actor, first keep actor records in `crew`, match those records to `person` using the person ID, and then match the credits to `movie` using the movie ID. Apply the name filter to `person` and sort the selected output columns at the end. Each merge corresponds to one join condition in SQL.

In [ ]:
# Find all movies ("official_title") where Anne Hathaway appears as an actress

In [ ]:
# SQL:
# SELECT person.given_name, movie.official_title, movie.year
# FROM imdb.crew
# INNER JOIN imdb.person ON crew.person = person.id
# INNER JOIN imdb.movie ON crew.movie = movie.id
# WHERE crew.p_role = 'actor' AND person.given_name = 'Anne Hathaway'
# ORDER BY movie.year, movie.official_title;
actor_credits = (
    crew.loc[crew["p_role"] == "actor"]
    .merge(
        person[person["given_name"] == "Anne Hathaway"], left_on="person", right_on="id"
    )
    .merge(movie, left_on="movie", right_on="id", suffixes=("_person", "_movie"))
    .loc[:, ["given_name", "official_title", "year"]]
    .sort_values(["year", "official_title"])
)

actor_credits

### Derive a value after joining

A movie may have multiple rating records, for example from different sources or check dates. After joining `movie` and `rating`, create a normalized score with `score / scale`. A score of 8 on a scale of 10 has ratio `0.8`; comparing the ratio works across sources that use different scales.

In [ ]:
# Compute a normalized movie score ("score" / "scale"), and only select movies ("official_title") with ratio higher than 0.8

In [ ]:
# SQL:
# SELECT movie.id, movie.official_title, rating.score, rating.scale, rating.score / rating.scale AS score_ratio
# FROM imdb.movie
# INNER JOIN imdb.rating ON movie.id = rating.movie
# WHERE rating.score / rating.scale > 0.8;
movie_ratings = movie.merge(
    rating.assign(score_ratio=rating["score"] / rating["scale"]),
    left_on="id",
    right_on="movie"
)

movie_ratings.loc[
    movie_ratings["score_ratio"] > 0.8,
    ["id", "official_title", "score", "scale", "score_ratio"]
]

### Exercise 2 — Joins and distinct results

Return the **distinct movie titles** of movies produced in Italy (`ITA`). Use `produced` to identify the movie IDs, then use `movie` to retrieve the titles. Keep only the title column in the final result.

In [ ]:
italian_ids = produced.loc[produced["country"] == "ITA", "movie"]
movie.loc[movie["id"].isin(italian_ids), ["official_title"]].drop_duplicates()

## 9. Left joins and missing relationships (anti-joins)

A `LEFT JOIN` preserves every row from the left table, including rows without a match. In SQL, unmatched rows are often found with `WHERE right_table.key IS NULL`.

In Pandas, `merge(..., how="left", indicator=True)` adds a `_merge` column. It contains `left_only` for rows with no match and `both` for matched rows. Filtering for `left_only` is a clear way to implement a common **anti-join**.

In [ ]:
# Find all the movies ("official_title") that have not been rated

In [ ]:
# SQL:
# SELECT m.id, m.official_title
# FROM imdb.movie AS m
# LEFT JOIN (SELECT DISTINCT movie FROM imdb.rating) AS r ON m.id = r.movie
# WHERE r.movie IS NULL;
unrated_movies = movie.merge(
    rating[["movie"]].drop_duplicates(),
    how="left",
    left_on="id",
    right_on="movie",
    indicator=True
)

unrated_movies.loc[unrated_movies["_merge"] == "left_only", ["id", "official_title"]]

### Exercise 3 — Find missing relationships

Find movies that have **no record in `genre`**. Return the movie ID and official title. Use either a left merge with an indicator or a membership test with `.isin()` and `~`.

In [ ]:
movie.loc[~movie["id"].isin(genre["movie"]), ["id", "official_title"]]

## 10. Set operations: `INTERSECT`, `EXCEPT`, `UNION`, and `UNION ALL`

SQL set operations combine compatible query results. For these one-column examples, Pandas can express the same logic with `.isin()`, Boolean masks, and `pd.concat()`.

In [ ]:
# Find movies (only ids will suffice) that have been produced: both in the U.S. and in France; in the U.S. but not in France; in the U.S. or in France

In [ ]:
# SQL:
# SELECT movie FROM imdb.produced WHERE country = 'USA';
usa_ids = produced.loc[produced["country"] == "USA", "movie"].drop_duplicates()

# SQL:
# SELECT movie FROM imdb.produced WHERE country = 'FRA';
france_ids = produced.loc[produced["country"] == "FRA", "movie"].drop_duplicates()

# SQL: SELECT COUNT(DISTINCT movie) FROM imdb.produced WHERE country = 'USA';
print("Distinct USA movie IDs:", len(usa_ids))
# SQL: SELECT COUNT(DISTINCT movie) FROM imdb.produced WHERE country = 'FRA';
print("Distinct France movie IDs:", len(france_ids))

In [ ]:
# SQL:
# (SELECT movie FROM imdb.produced WHERE country = 'USA')
# INTERSECT
# (SELECT movie FROM imdb.produced WHERE country = 'FRA');
movies_in_both = usa_ids[usa_ids.isin(france_ids)].reset_index(drop=True)
movies_in_both

In [ ]:
# SQL:
# (SELECT movie FROM imdb.produced WHERE country = 'USA')
# EXCEPT
# (SELECT movie FROM imdb.produced WHERE country = 'FRA');
usa_only_ids = usa_ids[~usa_ids.isin(france_ids)].reset_index(drop=True)
usa_only_ids

`UNION ALL` preserves duplicate rows, while `UNION` removes duplicate result rows. Concatenate the two Series to simulate `UNION ALL`, then call `drop_duplicates()` to simulate `UNION`. Because each input Series was deduplicated by country, a movie produced in both countries appears twice in `UNION ALL` and once in `UNION`.

In [ ]:
# SQL:
# (SELECT movie FROM imdb.produced WHERE country = 'USA')
# UNION ALL
# (SELECT movie FROM imdb.produced WHERE country = 'FRA');
union_all = pd.concat([usa_ids, france_ids], ignore_index=True)
union_all

In [ ]:
# SQL:
# (SELECT movie FROM imdb.produced WHERE country = 'USA')
# UNION
# (SELECT movie FROM imdb.produced WHERE country = 'FRA');
union = union_all.drop_duplicates().reset_index(drop=True)
union

## 11. Aggregate values and group rows

Aggregation summarizes many rows into one or more values. Pandas Series methods correspond to SQL aggregate functions such as `MIN`, `MAX`, `AVG`, and `SUM`.

These counts have different meanings:

- SQL `COUNT(*)` counts rows, even if some column values are missing. In Pandas, use `len(dataframe)` or grouped `size`.
- SQL `COUNT(column)` counts non-`NULL` values. In Pandas, use `Series.count()` or grouped `count`.
- SQL `COUNT(DISTINCT column)` counts distinct non-`NULL` values. Pandas `Series.nunique()` excludes missing values by default.

In [ ]:
# SQL:
# SELECT MIN(length), MAX(length), AVG(length), SUM(length), COUNT(length), COUNT(DISTINCT length)
# FROM imdb.movie;
length_stats = movie["length"].agg(["min", "max", "mean", "sum", "count", "nunique"])
print("COUNT(*) equivalent:", len(movie))
length_stats

### Group by one column

`groupby()` splits rows into groups, then calculates a summary for each group. Use `size` when you need to count rows (`COUNT(*)`), and `mean` for the average of non-missing lengths.

**Note:** SQL `GROUP BY year` includes a group for `NULL` years. Pandas excludes rows whose group key is missing by default, so use `dropna=False` to keep that group. The correlated-query example later needs a different choice because SQL's `NULL = NULL` comparison does not match rows.

In [ ]:
# Compute the number of movies and their average length by year, and sort the output by movie count

In [ ]:
# SQL:
# SELECT year, COUNT(*) AS movie_count, AVG(length) AS average_length
# FROM imdb.movie
# GROUP BY year
# ORDER BY movie_count DESC;
year_summary = (
    movie.groupby(by="year", as_index=False, dropna=False)
    .agg(movie_count=("id", "size"), average_length=("length", "mean"))
    .sort_values("movie_count", ascending=False)
)

year_summary

### Group by multiple columns

Pass a list of grouping columns to `groupby()`. Here, each group corresponds to a movie-role pair, and `size` counts all rows in that group.

In [ ]:
# Count the number of people for each role in the cast of each movie, and sort the result by movie (id) and role

In [ ]:
# SQL:
# SELECT movie, p_role, COUNT(*) AS people_count
# FROM imdb.crew
# GROUP BY movie, p_role
# ORDER BY movie, p_role;
crew_counts = (
    crew.groupby(by=["movie", "p_role"], as_index=False, dropna=False)
    .agg(people_count=("person", "size"))
    .sort_values(["movie", "p_role"])
)

crew_counts

### `WHERE` versus `HAVING`

`WHERE` filters individual rows **before** aggregation; `HAVING` filters groups **after** aggregation. In Pandas, filter `crew` to actor records first, group those records by person, calculate a count, and then filter the summary DataFrame.

Here, the requirement is to count **distinct movies** per actor. That is why the Pandas version uses `nunique()` and SQL uses `COUNT(DISTINCT crew.movie)`, rather than counting raw credit rows.

In [ ]:
# Count the number of (unique) movies for each actor (reporting person id and name), then select only actors with more than 5 movies

In [ ]:
actor_counts = (
    crew.loc[crew["p_role"] == "actor"]
    .merge(person, left_on="person", right_on="id")
    .groupby(["person", "given_name"], as_index=False, dropna=False)
    .agg(movie_count=("movie", "nunique"))
)

actor_counts

In [ ]:
# SQL:
# SELECT crew.person, person.given_name, COUNT(DISTINCT crew.movie) AS movie_count
# FROM imdb.crew
# INNER JOIN imdb.person ON crew.person = person.id
# WHERE crew.p_role = 'actor'
# GROUP BY crew.person, person.given_name
# HAVING COUNT(DISTINCT crew.movie) > 5;
actor_counts = (
    crew.loc[crew["p_role"] == "actor"]
    .merge(person, left_on="person", right_on="id")
    .groupby(["person", "given_name"], as_index=False, dropna=False)
    .agg(movie_count=("movie", "nunique"))
)

actor_counts.loc[
    actor_counts["movie_count"] > 5, ["person", "given_name", "movie_count"]
]

### Exercise 4 — Grouping and `HAVING`

Return drama movies that have been produced in **more than one country**, together with the movie IDs, official titles, and number of distinct production countries. Filter to drama movies before grouping; filter the summary after calculating the country count.

In [ ]:
drama_movie_ids = genre.loc[genre["genre"].str.lower() == "drama", "movie"]
drama_productions = (
    produced.loc[produced["movie"].isin(drama_movie_ids)]
    .merge(movie[["id", "official_title"]], left_on="movie", right_on="id")
)

drama_country_counts = (
    drama_productions.groupby(["id", "official_title"], as_index=False)
    .agg(country_count=("country", "nunique"))
)

drama_country_counts[drama_country_counts["country_count"] > 1]

## 12. Subqueries and comparisons across groups

Some SQL queries compare each row with a value calculated from related rows. Pandas can calculate a group-level value and align it to each original row with `groupby(...).transform(...)`.

For example, a subquery can find movies whose length is greater than the average length for their release year. `transform("mean")` produces one average per row, so the comparison remains row by row.

In [ ]:
# Select all movies whose length is greater than the average movie length computed for their year of production

In [ ]:
# SQL:
# SELECT m1.id, m1.official_title, m1.year, m1.length
# FROM imdb.movie AS m1
# WHERE m1.length > (
#     SELECT AVG(m2.length)
#     FROM imdb.movie AS m2
#     WHERE m2.year = m1.year
# );
movies_with_average = movie.assign(
    year_average=movie.groupby(by="year")["length"].transform("mean")
)

movies_with_average.loc[
    movies_with_average["length"] > movies_with_average["year_average"],
    ["id", "official_title", "year", "length", "year_average"]
]

## 13. Existence and exclusion

A frequent query pattern is: “include records that have a relationship of type A and **no** relationship of type B.” Filtering a single joined row can be wrong: a movie might have several actors, and one actor's row may satisfy a condition while another actor's row should exclude the movie.

A reliable approach is to build the set of IDs that must be excluded, then remove every matching movie. In this schema, `location.d_role = 'B'` denotes a birthplace record. Therefore, the first example finds movies with **no actor whose birthplace is Italy**; it does not claim that the actor has Italian nationality.

In [ ]:
# Select all movies that are not starring any Italian actor

In [ ]:
# SQL:
# SELECT m.id, m.official_title
# FROM imdb.movie AS m
# WHERE NOT EXISTS (
#     SELECT 1
#     FROM imdb.crew AS c
#     INNER JOIN imdb.location AS l ON c.person = l.person
#     WHERE c.movie = m.id AND c.p_role = 'actor' AND l.d_role = 'B' AND l.country = 'ITA'
# );
italian_born_people = location.loc[
    (location["d_role"] == "B") & (location["country"] == "ITA"), "person"
]

movies_with_italian_born_actors = crew.loc[
    (crew["p_role"] == "actor") & crew["person"].isin(italian_born_people), "movie"
]

movies_without_italian_born_actors = movie.loc[
    ~movie["id"].isin(movies_with_italian_born_actors), ["id", "official_title"]
]

movies_without_italian_born_actors

### Movies produced only in Italy

This query requires two conditions: a movie **must** have an Italian production record, and it **must not** have a production record in any other country. Compute the two sets of movie IDs, then keep Italian IDs that do not appear among non-Italian production IDs.

In [ ]:
# Select all movies that have been produced in Italy, and nowhere else

In [ ]:
# SQL:
# SELECT m.id, m.official_title
# FROM imdb.movie AS m
# WHERE EXISTS (
#     SELECT 1 FROM imdb.produced AS p
#     WHERE p.movie = m.id AND p.country = 'ITA'
# )
# AND NOT EXISTS (
#     SELECT 1 FROM imdb.produced AS p
#     WHERE p.movie = m.id AND p.country <> 'ITA'
# );
italian_production_ids = produced.loc[produced["country"] == "ITA", "movie"]
non_italian_production_ids = produced.loc[produced["country"] != "ITA", "movie"]
italy_only_ids = italian_production_ids.loc[
    ~italian_production_ids.isin(non_italian_production_ids)
]

movie.loc[movie["id"].isin(italy_only_ids), ["id", "official_title"]]

## 14. Final practice

Solve each following task with Pandas:

5. Find countries that have no production records. Return their ISO code and name.
6. Find movies whose length is **greater than 60 minutes** and **less than** the average length of movies from the same year.
7. Find people who have **never acted in a movie produced or co-produced in the USA**.
8. *(Extension)* Find movies produced in the USA that have never been released in Italy.

## Solutions

In [ ]:
# Exercise 1
# SQL:
# SELECT official_title, year, length
# FROM imdb.movie
# WHERE year = '2010' AND length >= 90
# ORDER BY length DESC;
(
    movie.loc[
        (movie["year"] == "2010") & (movie["length"] >= 90),
        ["official_title", "year", "length"]
    ]
    .sort_values("length", ascending=False)
)

In [ ]:
# Exercise 2
# SQL:
# SELECT DISTINCT m.official_title
# FROM imdb.movie AS m
# INNER JOIN imdb.produced AS p ON m.id = p.movie
# WHERE p.country = 'ITA';
italian_ids = produced.loc[produced["country"] == "ITA", "movie"]
movie.loc[movie["id"].isin(italian_ids), ["official_title"]].drop_duplicates()

In [ ]:
# Exercise 3
# SQL:
# SELECT m.id, m.official_title
# FROM imdb.movie AS m
# LEFT JOIN imdb.genre AS g ON m.id = g.movie
# WHERE g.movie IS NULL;
movie.loc[~movie["id"].isin(genre["movie"]), ["id", "official_title"]]

In [ ]:
# Exercise 4
# SQL:
# SELECT m.id, m.official_title, COUNT(DISTINCT p.country) AS country_count
# FROM imdb.movie AS m
# INNER JOIN imdb.genre AS g ON m.id = g.movie
# INNER JOIN imdb.produced AS p ON m.id = p.movie
# WHERE LOWER(g.genre) = 'drama'
# GROUP BY m.id, m.official_title
# HAVING COUNT(DISTINCT p.country) > 1;
drama_movie_ids = genre.loc[genre["genre"].str.lower() == "drama", "movie"]
drama_productions = (
    produced.loc[produced["movie"].isin(drama_movie_ids)]
    .merge(movie[["id", "official_title"]], left_on="movie", right_on="id")
)

drama_country_counts = (
    drama_productions.groupby(["id", "official_title"], as_index=False)
    .agg(country_count=("country", "nunique"))
)

drama_country_counts[drama_country_counts["country_count"] > 1]

In [ ]:
# Exercise 5
# SQL:
# SELECT c.iso3, c.name
# FROM imdb.country AS c
# LEFT JOIN imdb.produced AS p ON c.iso3 = p.country
# WHERE p.country IS NULL;
country.loc[~country["iso3"].isin(produced["country"]), ["iso3", "name"]]

In [ ]:
# Exercise 6
# SQL:
# SELECT m1.id, m1.official_title, m1.year, m1.length
# FROM imdb.movie AS m1
# WHERE m1.length > 60 AND m1.length < (
#     SELECT AVG(m2.length)
#     FROM imdb.movie AS m2
#     WHERE m2.year = m1.year
# );
movie_with_year_mean = movie.assign(
    year_average=movie.groupby(by="year")["length"].transform("mean")
)

movie_with_year_mean.loc[
    movie_with_year_mean["length"].between(
        60, movie_with_year_mean["year_average"], inclusive="neither"
    ),
    ["id", "official_title", "year", "length", "year_average"]
]

In [ ]:
# Exercise 7
american_movies = produced.loc[produced["country"] == "USA", "movie"]
actors_in_american_movies = crew.loc[
    (crew["p_role"] == "actor") & crew["movie"].isin(american_movies), "person"
]

person.loc[~person["id"].isin(actors_in_american_movies), ["id", "given_name"]]

In [ ]:
# Exercise 8 (Extension)
# SQL:
# SELECT DISTINCT m.id, m.official_title
# FROM imdb.movie AS m
# INNER JOIN imdb.produced AS p ON m.id = p.movie
# WHERE p.country = 'USA' AND NOT EXISTS (
#     SELECT 1 FROM imdb.released AS r
#     WHERE r.movie = m.id AND r.country = 'ITA'
# );
usa_production = produced.loc[produced["country"] == "USA", "movie"].drop_duplicates()
italian_release = released.loc[released["country"] == "ITA", "movie"].drop_duplicates()
usa_not_in_italy = usa_production[~usa_production.isin(italian_release)]

(
    movie.loc[movie["id"].isin(usa_not_in_italy), ["id", "official_title"]]
    .drop_duplicates()
)

## 16. Close resources

`engine.dispose()` releases SQLAlchemy's pooled connections.

In [ ]:
engine.dispose()